## MONAI Integration
MONAI allows the definition of AI models using the "bundle" concept. It allows for easy experimentation and sharing of models that have been developed using MONAI. Using the bundle configurations, we can use MONAI's MonaiAlgo (the implementation of ClientAlgo) to execute a bundle model in a federated scenario using NVFlare.

You can find the code for the [MONAI Integration code](https://github.com/NVIDIA/NVFlare/blob/dev/integration/monai/README.md) in the `integration` directory of the NVFlare GitHub.

There is also an example that walks through preparing the environment and input datasets, and running the [MONAI Spleen CT Segmentation example](https://github.com/NVIDIA/NVFlare/tree/dev/integration/monai/examples/spleen_ct_segmentation) using a locally provisioned secure deployment.  We'll walk through that here.

## Intro to Provisioning

In addition to the FL Simulator and POC modes, NVFlare ships with a provisioning system that can be used to generate scripts and configuration files for secure deployment of the FL server and clients.  These configurations also include certificates for establishing identity and secure communication between participants, and are packaged in unique subdirectory for each client and server.  We call these packages "startup kits".

To get started with the provisioning system, we can look at the `nvflare provision` command.


In [1]:
!nvflare provision -h

usage: nvflare provision [-h] [-p PROJECT_FILE] [-w WORKSPACE]
                         [-c CUSTOM_FOLDER] [--add_user ADD_USER]
                         [--add_client ADD_CLIENT]

optional arguments:
  -h, --help            show this help message and exit
  -p PROJECT_FILE, --project_file PROJECT_FILE
                        file to describe FL project
  -w WORKSPACE, --workspace WORKSPACE
                        directory used by provision
  -c CUSTOM_FOLDER, --custom_folder CUSTOM_FOLDER
                        additional folder to load python codes
  --add_user ADD_USER   yaml file for added user
  --add_client ADD_CLIENT
                        yaml file for added client


Provisioning is based on a PROJECT_FILE that defines the server and clients, and the set of NVFlare Builder modules that are used to generate the startup kits.  You can run `nvflare provision` without any arguments to generate a sample `project.yml` file.  This will prompt you to choose either a High-Availability (HA) or non-HA configuration.


### Provisioning for Docker Compose deployment

For this demo, we've already generated a `project.yml` config, choosing non-HA for simplicity, and have added an additional Builder module to generate a Docker compose environment for deploying locally.  This confiuguration is in [`project-non-HA.yml`](project-non-HA.yml) (click to open in an editor).  The DockerBuilder module was added following the WorkspaceBuilder config with:
```yaml
  - path: nvflare.lighter.impl.docker.DockerBuilder
    args:
      base_image: gtc-dli-nvflare-monai:latest
```

We can generate the project workspace by running `nvflare provision -p project-non-HA.yml`:


In [2]:
!nvflare provision -w monai_workspace -p project-non-HA.yml


Path list (sys.path) for python codes loading: ['/opt/conda/bin', '/opt/conda/lib/python38.zip', '/opt/conda/lib/python3.8', '/opt/conda/lib/python3.8/lib-dynload', '/opt/conda/lib/python3.8/site-packages', '/opt/conda/lib/python3.8/site-packages/torchtext-0.11.0a0-py3.8-linux-x86_64.egg', '/opt/conda/lib/python3.8/site-packages/certifi-2022.9.14-py3.8.egg', '/opt/conda/lib/python3.8/site-packages/functorch-0.3.0a0-py3.8-linux-x86_64.egg', '/flare/NVFlare', '/flare/notebooks', '/flare/notebooks/.'] 

Project yaml file: /flare/notebooks/project-non-HA.yml.
Generated results can be found under /flare/notebooks/monai_workspace/monai_demo/prod_00.  Builder's wip folder removed.


**_Note:_** Because we're running in a container, we need make a few adjustments to the Docker compose config generated during provision, and then copy the workspace to a location accessible by the host system.  *This is not necessary when running on a local system.*

In [3]:
# Fix the Docker compose setup for running within the container
%env HOST_PATH=/tmp/monai_workspace/monai_demo/prod_00
!echo HOST_PATH=/tmp/monai_workspace/monai_demo/prod_00 >> monai_workspace/monai_demo/prod_00/.env
!sed -i s,nvflare-service,gtc-dli-nvflare-monai,g monai_workspace/monai_demo/prod_00/.env
!sed -i s,/usr/local/bin/python3,/opt/conda/bin/python3,g monai_workspace/monai_demo/prod_00/.env
!sed -i s,./server1,\$\{HOST_PATH\}/server1,g monai_workspace/monai_demo/prod_00/compose.yaml
!sed -i s,./site-1,\$\{HOST_PATH\}/site-1,g monai_workspace/monai_demo/prod_00/compose.yaml
!sed -i s,./site-2,\$\{HOST_PATH\}/site-2,g monai_workspace/monai_demo/prod_00/compose.yaml
!cp -rf monai_workspace /tmp/monai_workspace

env: HOST_PATH=/tmp/monai_workspace/monai_demo/prod_00


### Launching the FL system
Now that we have our provisioning `workspace` directory with the `monai_demo/prod_00` configuration, we can launch the FL server and clients each in a docker container by using the `compose.yml` and `.env` that were generated during provisioning.

In [14]:
!docker compose --file monai_workspace/monai_demo/prod_00/compose.yaml --env-file monai_workspace/monai_demo/prod_00/.env up -d

[+] Running 0/0
 ⠋ Network prod_00_default  Creating                                       0.1s
[+] Running 1/1
 ⠿ Network prod_00_default  Created                                        0.1s
 ⠋ Container site-1         Creating                                       0.1s
 ⠋ Container site-2         Creating                                       0.1s
 ⠋ Container server1        Creating                                       0.1s
[+] Running 1/4
 ⠿ Network prod_00_default  Created                                        0.1s
 ⠙ Container site-1         Creating                                       0.2s
 ⠙ Container site-2         Creating                                       0.2s
 ⠙ Container server1        Creating                                       0.2s
[+] Running 1/4
 ⠿ Network prod_00_default  Created                                        0.1s
 ⠿ Container site-1         Starting                                       0.3s
 ⠿ Container site-2         Starting                    

We can verify that the `server1` and two clients, `site-1` and `site-2` are each running in a container instance (we will also see a fourth container `nvflare-monai` that's hosting this notebook and DLI content.)

In [5]:
!docker ps

CONTAINER ID   IMAGE                   COMMAND                  CREATED         STATUS         PORTS                                                                               NAMES
b5dd54cca3dc   gtc-dli-nvflare-monai   "/opt/docker/entrypo…"   6 seconds ago   Up 5 seconds   6006/tcp, 8888/tcp                                                                  site-2
bc441efc2b99   gtc-dli-nvflare-monai   "/opt/docker/entrypo…"   6 seconds ago   Up 5 seconds   6006/tcp, 8888/tcp, 0.0.0.0:8002-8003->8002-8003/tcp, :::8002-8003->8002-8003/tcp   server1
e32a9f960c6f   gtc-dli-nvflare-monai   "/opt/docker/entrypo…"   6 seconds ago   Up 5 seconds   6006/tcp, 8888/tcp                                                                  site-1
dde9889173f2   gtc-dli-nvflare-monai   "/opt/docker/entrypo…"   2 minutes ago   Up 2 minutes                                                                                       nvflare-monai


## Setting up the MONAI example

The MONAI example for a distributed (or local Docker Compose "distriburted" deployment) can be found in the `NVFlare/integration/monai/examples/spleen_ct_segmentation_real-world` directory.    Let's copy it to our `notebooks/examples` directory.

In [6]:
!if [ ! -d examples ]; then mkdir examples; fi
!if [ ! -d examples/spleen_ct_segmentation_real-world ]; then \
    cp -r ../NVFlare/integration/monai/examples/spleen_ct_segmentation_real-world examples/; fi
!tree examples/spleen_ct_segmentation_real-world

examples/spleen_ct_segmentation_real-world
├── README.md
├── download_spleen_dataset.py
├── job
│   ├── app
│   │   ├── config
│   │   │   ├── config_fed_client.json
│   │   │   ├── config_fed_server.json
│   │   │   ├── spleen_ct_segmentation
│   │   │   │   ├── LICENSE
│   │   │   │   ├── configs
│   │   │   │   │   ├── evaluate.json
│   │   │   │   │   ├── inference.json
│   │   │   │   │   ├── logging.conf
│   │   │   │   │   ├── metadata.json
│   │   │   │   │   ├── multi_gpu_evaluate.json
│   │   │   │   │   ├── multi_gpu_train.json
│   │   │   │   │   └── train.json
│   │   │   │   ├── docs
│   │   │   │   │   ├── README.md
│   │   │   │   │   └── data_license.txt
│   │   │   │   └── models
│   │   │   │       ├── model.pt
│   │   │   │       └── model.ts
│   │   │   └── spleen_ct_segmentation_v0.3.7.zip
│   │   └── configp
│   │       ├── spleen_ct_segmentation
│   │       │   ├── LICENSE
│   │       │   ├── configs
│   │       │   │   ├── evaluate.json
│   │       │   │   ├── 

We now need to download the MONAI bundle that contains the `spleen_ct_segmentation` model and configuration.  We can do this using the MONAI bundle download built-in script.

This model and configuration will be part of the FLARE spleen_ct_segmentation_real-world app that is deployed to the FLARE clients, so we'll provde the job directory as the bundle download path.

We will also download the example spleen dataset and push it to a directory accessible by the containers running the FLARE clients.

In [7]:
%env JOB_DIR=examples/spleen_ct_segmentation_real-world/job
!python3 -m monai.bundle download \
    --name "spleen_ct_segmentation" \
    --version "0.3.7" \
    --bundle_dir ./${JOB_DIR}/app/config
!if [ ! -d data/Task09_Spleen ]; then \
    python3 examples/spleen_ct_segmentation_real-world/download_spleen_dataset.py; fi
!for site in site-1 site-2; do mkdir ${HOST_PATH}/${site}/data; cp -r data/Task09_Spleen ${HOST_PATH}/${site}/data; done

env: JOB_DIR=examples/spleen_ct_segmentation_real-world/job
2023-03-08 23:01:15,153 - INFO - --- input summary of monai.bundle.scripts.download ---
2023-03-08 23:01:15,153 - INFO - > name: 'spleen_ct_segmentation'
2023-03-08 23:01:15,153 - INFO - > version: '0.3.7'
2023-03-08 23:01:15,153 - INFO - > bundle_dir: './examples/spleen_ct_segmentation_real-world/job/app/config'
2023-03-08 23:01:15,153 - INFO - > source: 'github'
2023-03-08 23:01:15,153 - INFO - > repo: 'Project-MONAI/model-zoo/hosting_storage_v1'
2023-03-08 23:01:15,153 - INFO - > progress: True
2023-03-08 23:01:15,153 - INFO - ---


2023-03-08 23:01:15,153 - INFO - Expected md5 is None, skip md5 check for file examples/spleen_ct_segmentation_real-world/job/app/config/spleen_ct_segmentation_v0.3.7.zip.
2023-03-08 23:01:15,153 - INFO - File exists: examples/spleen_ct_segmentation_real-world/job/app/config/spleen_ct_segmentation_v0.3.7.zip, skipped downloading.
2023-03-08 23:01:15,153 - INFO - Writing into directory: examples/

### Connecting to the Docker Compose deployment and running the MONAI app
Even though we're running locally via Docker compose, the server and all clients are running securely in their own container instance.  Compared to POC mode, where we were running without authentication enabled, we will need to use the `admin@nvidia.com` toolkit to establish a secure session, providing the admin username `admin@nvidia.com` and the certificates provided in the `monai_workspace/monai_demo/prod_00/admin@nvidia.com` directory.

In [15]:
admin_dir = "monai_workspace/monai_demo/prod_00/admin@nvidia.com"
admin_user = "admin@nvidia.com"
from nvflare.fuel.flare_api.flare_api import new_secure_session

admin_session = new_secure_session(
    username = admin_user,
    startup_kit_location = admin_dir
)
print(admin_session.get_system_info())

SystemInfo
server_info:
status: stopped, start_time: Wed Mar  8 23:28:24 2023
client_info:
site-2(last_connect_time: Wed Mar  8 23:29:06 2023)
site-1(last_connect_time: Wed Mar  8 23:29:06 2023)
job_info:



In [17]:
path_to_job_config = "/flare/notebooks/examples/spleen_ct_segmentation_real-world/job"
job_id = admin_session.submit_job(path_to_job_config)
print(job_id)

87eaeabf-136f-4063-8833-d9ca72e0eab9


In [18]:
import json

# Job Status
jobs_output = admin_session.list_jobs()
jobs_detail = admin_session.list_jobs(detailed=True)
print("Job Status")
print(json.dumps((jobs_output), indent=2))
print("\nJob Detail")
print(json.dumps((jobs_detail), indent=2))

# Job Metadata
print("\nJob Metadata")
admin_session.get_job_meta(job_id)

Job Status
[
  {
    "job_id": "87eaeabf-136f-4063-8833-d9ca72e0eab9",
    "job_name": "spleen-bundle",
    "status": "RUNNING",
    "submit_time": "2023-03-08T23:29:55.502976+00:00",
    "duration": "0:00:00.420204"
  },
  {
    "job_id": "91f01d3a-3802-4fab-bcc5-4979aa4bd64d",
    "job_name": "spleen-bundle",
    "status": "FINISHED:EXECUTION_EXCEPTION",
    "submit_time": "2023-03-08T23:06:37.878331+00:00",
    "duration": "0:01:41.972576"
  }
]

Job Detail
[
  {
    "name": "spleen-bundle",
    "resource_spec": {},
    "min_clients": 2,
    "deploy_map": {
      "app": [
        "@ALL"
      ]
    },
    "job_folder_name": "job",
    "submitter_name": "admin@nvidia.com",
    "submitter_org": "nvidia",
    "submitter_role": "project_admin",
    "job_id": "87eaeabf-136f-4063-8833-d9ca72e0eab9",
    "submit_time": 1678318195.5029764,
    "submit_time_iso": "2023-03-08T23:29:55.502976+00:00",
    "start_time": "2023-03-08 23:29:58.563456",
    "duration": "0:00:00.450473",
    "status"

{'name': 'spleen-bundle',
 'resource_spec': {},
 'min_clients': 2,
 'deploy_map': {'app': ['@ALL']},
 'job_folder_name': 'job',
 'submitter_name': 'admin@nvidia.com',
 'submitter_org': 'nvidia',
 'submitter_role': 'project_admin',
 'job_id': '87eaeabf-136f-4063-8833-d9ca72e0eab9',
 'submit_time': 1678318195.5029764,
 'submit_time_iso': '2023-03-08T23:29:55.502976+00:00',
 'start_time': '2023-03-08 23:29:58.563456',
 'duration': 'N/A',
 'status': 'RUNNING',
 'job_deploy_detail': ['server: OK', 'site-2: OK', 'site-1: OK'],
 'schedule_count': 1,
 'last_schedule_time': 1678318196.3851736,
 'schedule_history': ['2023-03-08 23:29:56: scheduled']}